In [17]:
import os
import json
from typing import cast
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from openai.types.responses import ResponseInputParam
from openai.types.responses import FunctionToolParam





#Initialize

load_dotenv(override=True)


openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL ="gpt-4o-mini"

openai = OpenAI()


In [ ]:
response = openai.responses.create(
    model="gpt-4o-mini",
    input="Write a one-sentence bedtime story about a unicorn.",
)

print(response.output_text)

In [6]:

system_message ="you are a helpful assistant"
context: ResponseInputParam = [{"role":"system","content":system_message}]

def stream_chat(message):
    context.append({"role":"user","content":message})
    stream = openai.responses.create(
    model=MODEL, 
    input=context, 
    stream=True
)
    
    chunk =""
    for event in stream:
        if event.type == "response.output_text.delta":
            chunk += event.delta
            print(chunk)


In [ ]:
def stream_chat2(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = history + [{"role":"user","content":message}]
    stream = openai.responses.create(
    model=MODEL, 
    input=message, 
    stream=True,
    store=False
)
    
    chunk =""
    for event in stream:
        if event.type == "response.output_text.delta":
            chunk += event.delta
            yield(chunk)

In [ ]:
gr.ChatInterface(fn=stream_chat2).launch()

## Tool Call

In [3]:
books_list =[
  {"book": "Muqaddimah", "author": "Ibn Khaldun"},
  {"book": "Al-Tahafut al-Tahafut", "author": "Al-Ghazali"},
  {"book": "Kitab al-Shifa", "author": "Avicenna (Ibn Sina)"},
  {"book": "Al-Muwatta", "author": "Imam Malik"},
  {"book": "Sahih al-Bukhari", "author": "Al-Bukhari"},
  {"book": "Al-Risala", "author": "Al-Shafi'i"},
  {"book": "Hayy ibn Yaqzan", "author": "Ibn Tufail"},
  {"book": "Fuṣūḥ al-Hikma", "author": "Rumi"},
  {"book": "Diwan al-Mutanabbi", "author": "Al-Mutanabbi"},
  {"book": "Al-Kamil fi al-Tarikh", "author": "Ibn Kathir"}
]

In [19]:
def check_books(book_name):
    if book_name in [book["book"] for book in books_list]:
        return "the book is in stock"
    else:
        return "the book is out of stock"

In [20]:
tools = [
    {
        "type": "function",
        "name": "check_books",
        "description": "check if a book is in stock or not",
        "parameters": {
            "type": "object",
            "properties": {
                "book_name": {
                    "type": "string",
                    "description": "name of the book that the customer is intereseted in",
                },
            },
            "required": ["book_name"],
        },
    }
]



"""
# Example call with the tool definition
response = openai.responses.create(
    model=MODEL,
    input="Check whether the book 'Kitab al-Shifa' is in the list.",
    tools=[book_lookup_tool],
    tool_choice="auto"
)

print(response)

# If the model calls the tool, inspect the response output
for item in response.output:
    if item.type == "function_call":
        print("Tool called:", item.name)
        print("Arguments:", item.arguments)
"""


'\n# Example call with the tool definition\nresponse = openai.responses.create(\n    model=MODEL,\n    input="Check whether the book \'Kitab al-Shifa\' is in the list.",\n    tools=[book_lookup_tool],\n    tool_choice="auto"\n)\n\nprint(response)\n\n# If the model calls the tool, inspect the response output\nfor item in response.output:\n    if item.type == "function_call":\n        print("Tool called:", item.name)\n        print("Arguments:", item.arguments)\n'

# using no stream

In [23]:
input_list = [{"role":"user","content":"I'm looking for a book named Muqaddimah"}]

typed_tools = cast(list[FunctionToolParam], tools)

In [ ]:



response = openai.responses.create(
    model=MODEL,
    input=cast(ResponseInputParam, input_list),
    store=False,
    tools=typed_tools,
)


input_list += response.output


for item in response.output:
    if item.type == "function_call" and item.name == "check_books":
        book_name = json.loads(item.arguments)["book_name"]
        book_available = check_books(book_name)

        input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": str(book_available),
                }
            )

print("Final input:")
print(input_list)


response = openai.responses.create(
    model=MODEL,
    input= cast(ResponseInputParam,input_list),
    store = False,
    tools = typed_tools

)

print(response.output_text)
        

Final input:
[{'role': 'user', 'content': "I'm looking for a book named Muqaddimah"}, ResponseFunctionToolCall(arguments='{"book_name":"Muqaddimah"}', call_id='call_bKeAru65gIK5fgYNIcaoW8SF', name='check_books', type='function_call', id='fc_0faac9ec86af3b64016aa9b0a1704887d2a56dced28f69f55d', status='completed'), {'type': 'function_call_output', 'call_id': 'call_bKeAru65gIK5fgYNIcaoW8SF', 'output': 'the book is in stock'}]
The book "Muqaddimah" is in stock! Would you like to know more about it or make a purchase?


# with streaming

In [25]:
response = openai.responses.create(
    model=MODEL,
    input = cast(ResponseInputParam, input_list),
    stream=True,
    store = False,
    tools = typed_tools
)


for event in response:
    print(event)

ResponseCreatedEvent(response=Response(id='resp_08eceb09451f3b85016aa9b567963c87d2807cdbc309988d9c', created_at=1789506919.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='check_books', parameters={'type': 'object', 'properties': {'book_name': {'type': 'string', 'description': 'name of the book that the customer is intereseted in'}}, 'required': ['book_name'], 'additionalProperties': False}, strict=True, type='function', description='check if a book is in stock or not', output_schema=None)], top_p=1.0, background=False, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None, context=None), safety_identifier=None, service_tier='auto', status='in_progress', text=ResponseTextConf